# Analyze AIT Log v2 Attack Phases

In [112]:
import pandas as pd

from utils import analyze_phase

## Load MSA Data

In [113]:
dataset = "aitv2"
scenario = "santos"
data_dir = f"../data/interim/{dataset}/{scenario}/flows_labeled"
file_path = f"{data_dir}/all_flows_labeled.csv"

In [114]:
df = pd.read_csv(file_path)

In [115]:
df.head()

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase
0,f1115,1.642118e+09,1.642118e+09,0.021806,172.21.128.54,55156,192.168.104.218,445,tcp,gssapi,...,514,4,379,-,6,benign_share,cloud_share,2022-01-14 00:00:02.425734043,2022-01-14 00:00:02.447540045,0
1,f3100,1.642118e+09,1.642118e+09,0.019958,172.21.128.54,55156,192.168.104.218,445,tcp,gssapi,...,514,4,379,-,6,benign_share,internal_share,2022-01-14 00:00:02.427603960,2022-01-14 00:00:02.447561979,0
2,f1116,1.642118e+09,1.642118e+09,0.011838,172.21.128.54,55158,192.168.104.218,445,tcp,gssapi,...,514,4,379,-,6,benign_share,cloud_share,2022-01-14 00:00:02.512283087,2022-01-14 00:00:02.524121046,0
3,f3101,1.642118e+09,1.642118e+09,0.010949,172.21.128.54,55158,192.168.104.218,445,tcp,gssapi,...,514,4,379,-,6,benign_share,internal_share,2022-01-14 00:00:02.513200997,2022-01-14 00:00:02.524149895,0
4,f3102,1.642118e+09,1.642118e+09,0.005656,192.168.104.218,35467,192.168.104.1,53,udp,dns,...,296,1,312,-,17,benign DNS,internal_share,2022-01-14 00:00:09.728241920,2022-01-14 00:00:09.733897924,0


In [116]:
df.value_counts("phase")

phase
0    333964
1     12861
2      5670
4       228
3        26
Name: count, dtype: int64

## Per Phase Analysis

In [117]:
df_benign = df[df["phase"] == 0]
df_phase_1 = df[df["phase"] == 1]
df_phase_2 = df[df["phase"] == 2]
df_phase_3 = df[df["phase"] == 3]
df_phase_4 = df[df["phase"] == 4]   

### Phase 1 - Data Exfiltration

In [118]:
phase_1_labels = df_phase_1["label"].value_counts()
print("Phase 1 Label Distribution:")
print(phase_1_labels)

Phase 1 Label Distribution:
label
data exfiltration    12861
Name: count, dtype: int64


In [119]:
analyze_phase(df_phase_1, "Phase 1")

Total Flows: 12861

 --- IP distribution ---

Source IPs (1):
src_ip
10.229.255.254    12861
Name: count, dtype: int64

Destination IPs (1):
dst_ip
10.229.2.216    12861
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (11598):
sport
1772     4
8931     4
6966     4
60954    3
62882    3
        ..
58934    1
18514    1
48402    1
25045    1
23702    1
Name: count, Length: 11598, dtype: int64

Destination Ports (1):
dport
53    12861
Name: count, dtype: int64


(src_ip
 10.229.255.254    12861
 Name: count, dtype: int64,
 dst_ip
 10.229.2.216    12861
 Name: count, dtype: int64,
 sport
 1772     4
 8931     4
 6966     4
 60954    3
 62882    3
         ..
 58934    1
 18514    1
 48402    1
 25045    1
 23702    1
 Name: count, Length: 11598, dtype: int64,
 dport
 53    12861
 Name: count, dtype: int64)

In [120]:
def analyze_origin_destination(df, phase_name):
    origins = df["local_orig"].value_counts()
    print(f"{phase_name} Origin Distribution:")
    print(origins)

    destinations = df["local_resp"].value_counts()
    print(f"{phase_name} Destination Distribution:")
    print(destinations)

In [121]:
analyze_origin_destination(df_phase_1, "Phase 1")

Phase 1 Origin Distribution:
local_orig
T    12861
Name: count, dtype: int64
Phase 1 Destination Distribution:
local_resp
T    12861
Name: count, dtype: int64


### Phase 2 Analysis - Scanning

In [122]:
print("Number of samples in Phase 2:", len(df_phase_2))

Number of samples in Phase 2: 5670


In [123]:
analyze_origin_destination(df_phase_2, "Phase 2")

Phase 2 Origin Distribution:
local_orig
T    5670
Name: count, dtype: int64
Phase 2 Destination Distribution:
local_resp
T    5670
Name: count, dtype: int64


In [124]:
analyze_phase(df_phase_2, "Phase 2")

Total Flows: 5670

 --- IP distribution ---

Source IPs (2):
src_ip
172.21.128.119    5074
10.229.2.216       596
Name: count, dtype: int64

Destination IPs (263):
dst_ip
192.168.104.155    1035
172.21.131.50       896
172.21.128.54       870
192.168.104.218     841
10.229.255.254      596
                   ... 
192.168.104.95        1
192.168.104.198       1
192.168.104.141       1
192.168.104.81        1
192.168.104.98        1
Name: count, Length: 263, dtype: int64

 --- Port distribution ---
Source Ports (3807):
sport
41554    6
60056    6
49810    5
56148    5
44526    5
        ..
47190    1
54112    1
53852    1
41092    1
34884    1
Name: count, Length: 3807, dtype: int64

Destination Ports (592):
dport
443     834
80      589
587      26
993      14
139      14
       ... 
146       5
8873      5
5633      5
800       5
1088      5
Name: count, Length: 592, dtype: int64


(src_ip
 172.21.128.119    5074
 10.229.2.216       596
 Name: count, dtype: int64,
 dst_ip
 192.168.104.155    1035
 172.21.131.50       896
 172.21.128.54       870
 192.168.104.218     841
 10.229.255.254      596
                    ... 
 192.168.104.95        1
 192.168.104.198       1
 192.168.104.141       1
 192.168.104.81        1
 192.168.104.98        1
 Name: count, Length: 263, dtype: int64,
 sport
 41554    6
 60056    6
 49810    5
 56148    5
 44526    5
         ..
 47190    1
 54112    1
 53852    1
 41092    1
 34884    1
 Name: count, Length: 3807, dtype: int64,
 dport
 443     834
 80      589
 587      26
 993      14
 139      14
        ... 
 146       5
 8873      5
 5633      5
 800       5
 1088      5
 Name: count, Length: 592, dtype: int64)

In [125]:
df_phase_2.columns

Index(['flow_id', 'start_time', 'end_time', 'duration', 'src_ip', 'sport',
       'dst_ip', 'dport', 'proto', 'service', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents', 'ip_proto', 'label', 'sensor_host', 'start_time_dt',
       'end_time_dt', 'phase'],
      dtype='object')

In [126]:
phase_2_labels = df_phase_2["label"].value_counts()
print("Phase 2 Label Distribution:")
print(phase_2_labels)

Phase 2 Label Distribution:
label
service_scan             4056
host_discover_local      1000
dns_brute_force_start     325
host_discover_dmz         115
dirb_scan                  90
wpscan                     84
Name: count, dtype: int64


In [127]:
phase_2_labels = df_phase_2["label"].unique()
print("Unique labels in Phase 2:", phase_2_labels)

Unique labels in Phase 2: ['host_discover_dmz' 'dns_brute_force_start' 'host_discover_local'
 'service_scan' 'dirb_scan' 'wpscan']


In [128]:
phase_2_protocols = df_phase_2["proto"].value_counts()
print("Phase 2 Protocol Distribution:")
print(phase_2_protocols)

Phase 2 Protocol Distribution:
proto
tcp    5670
Name: count, dtype: int64


In [129]:
phase_2_ports = df_phase_2["dport"].value_counts()
print("Phase 2 Destination Port Distribution:")
print(phase_2_ports)

Phase 2 Destination Port Distribution:
dport
443     834
80      589
587      26
993      14
139      14
       ... 
146       5
8873      5
5633      5
800       5
1088      5
Name: count, Length: 592, dtype: int64


In [130]:
# Label = host discover local
curr_label = phase_2_labels[0]
print(f"Analyzing Phase 2 - {curr_label}")
df_host_discover_local = df_phase_2[df_phase_2["label"] == curr_label]
analyze_origin_destination(df_host_discover_local, f"Phase 2 - {curr_label}")
analyze_phase(df_host_discover_local, f"Phase 2 - {curr_label}")

Analyzing Phase 2 - host_discover_dmz
Phase 2 - host_discover_dmz Origin Distribution:
local_orig
T    115
Name: count, dtype: int64
Phase 2 - host_discover_dmz Destination Distribution:
local_resp
T    115
Name: count, dtype: int64
Total Flows: 115

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    115
Name: count, dtype: int64

Destination IPs (6):
dst_ip
172.21.129.224    98
172.21.131.122     6
172.21.128.54      4
172.21.128.2       4
172.21.131.50      2
172.21.128.1       1
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (60):
sport
35596    4
39150    3
34464    2
37488    2
48880    2
36224    2
48882    2
35104    2
58794    2
32924    2
54296    2
56452    2
35554    2
35556    2
48884    2
48676    2
35558    2
35560    2
35552    2
35550    2
48890    2
48888    2
35570    2
35568    2
35566    2
48886    2
35564    2
35562    2
35582    2
48898    2
35584    2
35586    2
35588    2
35590    2
35592    2
35594    2
48892    2
48894    2


(src_ip
 172.21.128.119    115
 Name: count, dtype: int64,
 dst_ip
 172.21.129.224    98
 172.21.131.122     6
 172.21.128.54      4
 172.21.128.2       4
 172.21.131.50      2
 172.21.128.1       1
 Name: count, dtype: int64,
 sport
 35596    4
 39150    3
 34464    2
 37488    2
 48880    2
 36224    2
 48882    2
 35104    2
 58794    2
 32924    2
 54296    2
 56452    2
 35554    2
 35556    2
 48884    2
 48676    2
 35558    2
 35560    2
 35552    2
 35550    2
 48890    2
 48888    2
 35570    2
 35568    2
 35566    2
 48886    2
 35564    2
 35562    2
 35582    2
 48898    2
 35584    2
 35586    2
 35588    2
 35590    2
 35592    2
 35594    2
 48892    2
 48894    2
 48896    2
 35572    2
 35574    2
 35576    2
 35578    2
 35580    2
 35598    2
 35600    2
 35602    2
 35604    2
 35606    2
 35608    2
 35610    2
 35612    2
 44198    1
 46436    1
 39822    1
 58320    1
 37884    1
 35922    1
 39992    1
 42112    1
 Name: count, dtype: int64,
 dport
 443    94


In [131]:
# Label = host discover local
df_host_discover_local = df_phase_2[df_phase_2["label"] == "host_discover_local"]
# analyze_origin_destination(df_host_discover_local, "Phase 2 - Host Discover Local")
analyze_phase(df_host_discover_local, "Phase 2 - Host Discover Local")

Total Flows: 1000

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    1000
Name: count, dtype: int64

Destination IPs (256):
dst_ip
192.168.104.205    12
192.168.104.3       4
192.168.104.6       4
192.168.104.7       4
192.168.104.8       4
                   ..
192.168.104.81      1
192.168.104.98      1
192.168.104.198     1
192.168.104.141     1
192.168.104.4       1
Name: count, Length: 256, dtype: int64

 --- Port distribution ---
Source Ports (950):
sport
42854    3
42672    3
46724    3
59342    3
40434    3
        ..
40794    1
47938    1
46144    1
35360    1
58640    1
Name: count, Length: 950, dtype: int64

Destination Ports (2):
dport
80     512
443    488
Name: count, dtype: int64


(src_ip
 172.21.128.119    1000
 Name: count, dtype: int64,
 dst_ip
 192.168.104.205    12
 192.168.104.3       4
 192.168.104.6       4
 192.168.104.7       4
 192.168.104.8       4
                    ..
 192.168.104.81      1
 192.168.104.98      1
 192.168.104.198     1
 192.168.104.141     1
 192.168.104.4       1
 Name: count, Length: 256, dtype: int64,
 sport
 42854    3
 42672    3
 46724    3
 59342    3
 40434    3
         ..
 40794    1
 47938    1
 46144    1
 35360    1
 58640    1
 Name: count, Length: 950, dtype: int64,
 dport
 80     512
 443    488
 Name: count, dtype: int64)

In [132]:
def attack_frequency(df, phase_name):
    time_range = df["end_time"].max() - df["start_time"].min()
    attack_count = len(df)
    frequency = attack_count / time_range if time_range > 0 else 0
    print(f"{phase_name} Attack Frequency: {frequency:.4f} attacks per second")

In [133]:
print(f"Analyzing Phase 2:\n")
for curr_label in phase_2_labels:
    print(f"=== {curr_label} ===")
    df_curr_label = df_phase_2[df_phase_2["label"] == curr_label]
    # analyze_origin_destination(df_curr_label, f"Phase 2 - {curr_label}")
    analyze_phase(df_curr_label, f"Phase 2 - {curr_label}")
    # attack_frequency(df_curr_label, f"Phase 2 - {curr_label}")
    print()

Analyzing Phase 2:

=== host_discover_dmz ===
Total Flows: 115

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    115
Name: count, dtype: int64

Destination IPs (6):
dst_ip
172.21.129.224    98
172.21.131.122     6
172.21.128.54      4
172.21.128.2       4
172.21.131.50      2
172.21.128.1       1
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (60):
sport
35596    4
39150    3
34464    2
37488    2
48880    2
36224    2
48882    2
35104    2
58794    2
32924    2
54296    2
56452    2
35554    2
35556    2
48884    2
48676    2
35558    2
35560    2
35552    2
35550    2
48890    2
48888    2
35570    2
35568    2
35566    2
48886    2
35564    2
35562    2
35582    2
48898    2
35584    2
35586    2
35588    2
35590    2
35592    2
35594    2
48892    2
48894    2
48896    2
35572    2
35574    2
35576    2
35578    2
35580    2
35598    2
35600    2
35602    2
35604    2
35606    2
35608    2
35610    2
35612    2
44198    1
46436    1
39822    1


### Phase 3 Analysis - Exploitation

In [134]:
phase_3_labels = df_phase_3["label"].unique()
print("Unique labels in Phase 3:", phase_3_labels)

Unique labels in Phase 3: ['upload_rce_shell' 'check_user_id' 'check_date' 'check_network_config'
 'check_ps_a' 'read_resolv' 'read_passwd' 'check_release'
 'check_netstat_t' 'list_web_dir' 'read_group' 'check_wp_config'
 'dump_wp_users']


In [135]:
for curr_label in phase_3_labels:
    print(f"Analyzing Phase 3 - {curr_label}")
    df_curr_label = df_phase_3[df_phase_3["label"] == curr_label]
    # analyze_origin_destination(df_curr_label, f"Phase 3 - {curr_label}")
    analyze_phase(df_curr_label, f"Phase 3 - {curr_label}")
    attack_frequency(df_curr_label, f"Phase 3 - {curr_label}")
    print(len(df_curr_label))
    print()

Analyzing Phase 3 - upload_rce_shell
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    2
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.104.155    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1):
sport
44674    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    2
Name: count, dtype: int64
Phase 3 - upload_rce_shell Attack Frequency: 0.6304 attacks per second
2

Analyzing Phase 3 - check_user_id
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    2
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.104.155    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1):
sport
44678    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    2
Name: count, dtype: int64
Phase 3 - check_user_id Attack Frequency: 20.9100 attacks per second
2

Analyzing Phase 3 - check_date
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.2

In [136]:
analyze_origin_destination(df_phase_3, "Phase 3")

Phase 3 Origin Distribution:
local_orig
T    26
Name: count, dtype: int64
Phase 3 Destination Distribution:
local_resp
T    26
Name: count, dtype: int64


In [137]:
analyze_phase(df_phase_3, "Phase 3")

Total Flows: 26

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    26
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.104.155    26
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (13):
sport
44674    2
44678    2
44686    2
44698    2
44704    2
44706    2
44712    2
44714    2
44716    2
44720    2
44726    2
44728    2
44730    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    26
Name: count, dtype: int64


(src_ip
 172.21.128.119    26
 Name: count, dtype: int64,
 dst_ip
 192.168.104.155    26
 Name: count, dtype: int64,
 sport
 44674    2
 44678    2
 44686    2
 44698    2
 44704    2
 44706    2
 44712    2
 44714    2
 44716    2
 44720    2
 44726    2
 44728    2
 44730    2
 Name: count, dtype: int64,
 dport
 443    26
 Name: count, dtype: int64)

### Phase 4 Analysis - Cracking

In [138]:
analyze_origin_destination(df_phase_4, "Phase 4")

Phase 4 Origin Distribution:
local_orig
T    228
Name: count, dtype: int64
Phase 4 Destination Distribution:
local_resp
T    228
Name: count, dtype: int64


In [139]:
analyze_phase(df_phase_4, "Phase 4")

Total Flows: 228

 --- IP distribution ---

Source IPs (4):
src_ip
172.21.128.119     222
10.229.0.4           3
10.229.1.118         2
192.168.104.155      1
Name: count, dtype: int64

Destination IPs (4):
dst_ip
172.21.129.224     182
192.168.104.155     40
10.229.2.216         3
10.229.2.25          3
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (114):
sport
44636    3
44732    2
44734    2
48904    2
44736    2
        ..
35800    2
48992    2
48990    2
48994    2
51280    1
Name: count, Length: 114, dtype: int64

Destination Ports (3):
dport
443    208
80      19
25       1
Name: count, dtype: int64


(src_ip
 172.21.128.119     222
 10.229.0.4           3
 10.229.1.118         2
 192.168.104.155      1
 Name: count, dtype: int64,
 dst_ip
 172.21.129.224     182
 192.168.104.155     40
 10.229.2.216         3
 10.229.2.25          3
 Name: count, dtype: int64,
 sport
 44636    3
 44732    2
 44734    2
 48904    2
 44736    2
         ..
 35800    2
 48992    2
 48990    2
 48994    2
 51280    1
 Name: count, Length: 114, dtype: int64,
 dport
 443    208
 80      19
 25       1
 Name: count, dtype: int64)

In [140]:
df_phase_4_protocols = df_phase_4["proto"].value_counts()
print("Phase 4 Protocol Distribution:")
print(df_phase_4_protocols)

Phase 4 Protocol Distribution:
proto
tcp    228
Name: count, dtype: int64


In [141]:
analyze_origin_destination(df_benign, "Benign")

Benign Origin Distribution:
local_orig
T    333964
Name: count, dtype: int64
Benign Destination Distribution:
local_resp
T    273029
F     60935
Name: count, dtype: int64


In [142]:
analyze_origin_destination(df, "All Phases")

All Phases Origin Distribution:
local_orig
T    352749
Name: count, dtype: int64
All Phases Destination Distribution:
local_resp
T    291814
F     60935
Name: count, dtype: int64


In [143]:
phase4_ips = set(df_phase_4["src_ip"]).union(set(df_phase_4["dst_ip"]))
print("Unique IPs in Phase 4:", len(phase4_ips))
print(phase4_ips)

Unique IPs in Phase 4: 7
{'172.21.128.119', '10.229.0.4', '172.21.129.224', '10.229.1.118', '10.229.2.25', '192.168.104.155', '10.229.2.216'}


In [144]:
df_phase_4[df_phase_4["dst_ip"] == "10.35.35.206"]
# {'172.17.130.196', '172.17.130.37', '192.168.255.254', '10.35.35.206', '192.168.128.4', '151.101.14.109', '192.168.130.77'}

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase


In [145]:
df_phase_4[df_phase_4["src_ip"] == "10.35.35.206"]

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase


In [146]:
def get_phase_ips(df, phase):
    print(f"=== Analyzing Phase {phase} IPs ===")
    phase_ips = set(df["src_ip"].unique()) | set(df["dst_ip"].unique())
    phase_src_ips = set(df["src_ip"].unique())
    phase_dst_ips = set(df["dst_ip"].unique())

    print(f"Unique IPs in Phase {phase}:", phase_ips)
    print(f"Unique Source IPs in Phase {phase}:", phase_src_ips)
    print(f"Unique Destination IPs in Phase {phase}:", phase_dst_ips)
    print()

    return phase_ips, phase_src_ips, phase_dst_ips

In [147]:
phase1_ips, phase1_src_ips, phase1_dst_ips = get_phase_ips(df_phase_1, 1)
phase2_ips, phase2_src_ips, phase2_dst_ips = get_phase_ips(df_phase_2, 2)
phase3_ips, phase3_src_ips, phase3_dst_ips = get_phase_ips(df_phase_3, 3)
phase4_ips, phase4_src_ips, phase4_dst_ips = get_phase_ips(df_phase_4, 4)

=== Analyzing Phase 1 IPs ===
Unique IPs in Phase 1: {'10.229.255.254', '10.229.2.216'}
Unique Source IPs in Phase 1: {'10.229.255.254'}
Unique Destination IPs in Phase 1: {'10.229.2.216'}

=== Analyzing Phase 2 IPs ===
Unique IPs in Phase 2: {'192.168.104.29', '192.168.104.210', '192.168.104.237', '192.168.104.127', '192.168.104.170', '192.168.104.148', '192.168.104.132', '172.21.129.224', '192.168.104.87', '192.168.104.67', '192.168.104.222', '192.168.104.21', '192.168.104.244', '192.168.104.231', '192.168.104.93', '192.168.104.128', '192.168.104.66', '192.168.104.95', '192.168.104.50', '192.168.104.162', '192.168.104.248', '192.168.104.89', '192.168.104.58', '192.168.104.60', '192.168.104.131', '192.168.104.113', '192.168.104.84', '192.168.104.223', '192.168.104.38', '192.168.104.107', '192.168.104.11', '192.168.104.55', '172.21.131.122', '192.168.104.100', '192.168.104.30', '192.168.104.175', '192.168.104.241', '192.168.104.59', '192.168.104.10', '192.168.104.146', '192.168.104.221

In [148]:
all_phase_ips = phase1_ips | phase2_ips | phase3_ips | phase4_ips
print(len(all_phase_ips), "unique IPs across all phases.")

for i, ip in enumerate(all_phase_ips):
    in_phases = []
    if ip in phase1_ips:
        in_phases.append("Phase 1")
    if ip in phase2_ips:
        in_phases.append("Phase 2")
    if ip in phase3_ips:
        in_phases.append("Phase 3")
    if ip in phase4_ips:
        in_phases.append("Phase 4")
    
    if len(in_phases) > 1:
        print(f"IP {ip} appears in: {', '.join(in_phases)}")

268 unique IPs across all phases.
IP 172.21.129.224 appears in: Phase 2, Phase 4
IP 172.21.128.119 appears in: Phase 2, Phase 3, Phase 4
IP 10.229.2.216 appears in: Phase 1, Phase 2, Phase 4
IP 10.229.255.254 appears in: Phase 1, Phase 2
IP 192.168.104.155 appears in: Phase 2, Phase 3, Phase 4
